In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_ventas_mes;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_ventas_mes
USING DELTA
COMMENT 'KPI Ventas mensuales'
AS
WITH base AS (

SELECT
    fa.id_venta,
    fa.cant,
    fa.total,
    fa.delivery,
    fa.vtaestado,
    f.mes
    
FROM catalog_ventas.gold.fact_ventas fa

LEFT JOIN catalog_ventas.gold.dim_fecha f
       ON fa.id_fecha = f.id_fecha

WHERE fa.vtaestado != 'anulado'

)

SELECT
    mes,
    --Total de Tickets
    COUNT(DISTINCT id_venta) AS total_tickets,

    --Facturación Total
    SUM(total)               AS facturacion,

    --Unidades Vendidas
    SUM(cant)                AS unidades_vendidas,

    --Ticket Promedio
    ROUND(SUM(total) / 100 /  COUNT(DISTINCT id_venta),2) AS ticket_promedio,

    --Unidades por Ticket
    ROUND(SUM(cant)/ 100 / COUNT(DISTINCT id_venta),2)  AS unidades_por_ticket,

    --% Delivery
    ROUND(COUNT(DISTINCT CASE WHEN delivery = TRUE THEN id_venta END)
        *100.00 / COUNT(DISTINCT id_venta),2 )AS pct_delivery,

    --Facturacion por canal
    SUM(CASE WHEN delivery = TRUE THEN total ELSE 0 END) AS fact_delivery,
    SUM(CASE WHEN delivery = FALSE THEN total ELSE 0 END) AS fact_local,

    -- % Facturacion por canal
  ROUND(
    SUM(CASE WHEN delivery = FALSE THEN total ELSE 0 END)
    * 100.0 / SUM(total)
  , 2) AS pct_fact_local,

  ROUND(
    SUM(CASE WHEN delivery = TRUE THEN total ELSE 0 END)
    * 100.0 / SUM(total)
  , 2) AS pct_fact_delivery,

    CURRENT_TIMESTAMP() AS _refresh_timestamp

FROM base
GROUP BY mes 
ORDER BY mes

In [0]:
%sql
SELECT * FROM catalog_ventas.gold.kpi_ventas_mes